<a href="https://colab.research.google.com/github/lp20590-star/Global-Job-Category-Imputation-Labor-Market-Intelligence/blob/main/Global_Job_Category_Imputation_Labor_Market_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Mount Google Drive and load the jobs CSV (Week 10)
from google.colab import drive
drive.mount("/content/drive")

import glob
import pandas as pd

found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found += glob.glob("/content/drive/MyDrive/" + pattern + "dfall_clean.csv")

if len(found) == 0:
    for pattern in ["", "*/", "*/*/", "*/*/*/"]:
        found += glob.glob("/content/drive/MyDrive/" + pattern + "sample_rows__3__629d0f3b*.csv")

if len(found) == 0:
    print("Can't find dfall_clean.csv in Google Drive.")
else:
    jobs = pd.read_csv(found[0])
    print(f"Loaded {len(jobs)} jobs from:", found[0])
    print(jobs.head(3))

Mounted at /content/drive
Loaded 4000 jobs from: /content/drive/MyDrive/MIST 4600/dfall_clean.csv
           id                                              title  \
0  5410474609                              Mechanical Pipefitter   
1  5381519011  International Trade Policy Editor [Startup Pol...   
2  5389371380                                   Domestic Cleaner   

             company_name                 category_label  \
0              Dodd Group      Trade & Construction Jobs   
1         Applio Ventures                  Graduate Jobs   
2  Maid2Clean Suffolk Ltd  Domestic help & Cleaning Jobs   

                  category_tag         country      location_display  \
0      trade-construction-jobs  United Kingdom  King's Lynn, Norfolk   
1                graduate-jobs  United Kingdom            London, UK   
2  domestic-help-cleaning-jobs  United Kingdom         Diss, Norfolk   

                                  location_area   latitude  longitude  \
0  UK > Eastern England > 

In [ ]:
# Step 2: Define our class, helper function, and Sub-Question 1 (Weeks 3-8, 10)
ALLOWED = [
    "healthcare & nursing",
    "engineering",
    "trade & construction",
    "it & software development",
    "sales & business development",
    "accounting, banking & finance",
    "logistics, driver & supply chain",
    "teaching & education",
    "hospitality & catering",
    "administrative & office support",
    "hr & recruitment",
    "marketing, pr & advertising",
    "customer service & support",
    "social work & care",
    "maintenance, cleaning & facility",
    "other / specialized",
]

def is_missing_category(tag):
    clean = str(tag).strip().lower()
    if clean in ["unknown", "other-general-jobs", "", "nan"]:
        return True
    return False

class JobPostingAnalyzer:
    def __init__(self, df):
        self.df = df
        self.countries = [
            "Germany", "Switzerland", "France", "Spain",
            "Netherlands", "Italy", "United Kingdom", "USA",
        ]

    def mark_missing(self):
        self.df["is_missing"] = self.df["category_tag"].apply(is_missing_category)
        return self.df

    def missing_rate(self, country):
        sub = self.df[self.df["country"] == country]
        if len(sub) == 0:
            return 0.0
        missing = sub[sub["is_missing"] == True]
        return round((len(missing) / len(sub)) * 100, 1)

def answer_subquestion_1(analyzer):
    df = analyzer.mark_missing()
    print("--- Sub-Question 1: Missing Category Rate by Country ---")
    for country in analyzer.countries:
        pct = analyzer.missing_rate(country)
        print(f"{country}: {pct}% missing")

    print("\n--- Mean Salary by Country (Local Currency) ---")
    print(df.groupby("country")["salary_mid"].mean().round(2))
    return df

analyzer = JobPostingAnalyzer(jobs)
jobs = answer_subquestion_1(analyzer)

--- Sub-Question 1: Missing Category Rate by Country ---
Germany: 86.8% missing
Switzerland: 71.0% missing
France: 65.0% missing
Spain: 64.6% missing
Netherlands: 56.2% missing
Italy: 31.8% missing
United Kingdom: 0.4% missing
USA: 0.0% missing

--- Mean Salary by Country (Local Currency) ---
country
France            123224.47
Germany            23114.74
Italy              19035.11
Netherlands        34016.83
Spain               7019.12
Switzerland        31469.00
USA                74163.67
United Kingdom     39405.11
Name: salary_mid, dtype: float64


In [ ]:
# Step 3: Outside Tool #1 — Classify titles with Gemini API (with simple retry if Google 503 busy)
import time
import pandas as pd
from google import genai
from google.colab import userdata

key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=key)

def get_job_category(title, client):
    prompt = f"Pick ONE category from {ALLOWED}. Reply with only the category name in lowercase. Job: {title}"
    answer = ""
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt,
                config={"thinking_config": {"thinking_level": "LOW"}},
            )
            if response.text:
                answer = response.text.strip().lower()
            break
        except Exception:
            print("Gemini server busy (503), retrying in 5 seconds...")
            time.sleep(5)

    result = "other / specialized"
    for cat in ALLOWED:
        if cat in answer:
            result = cat

    print("Classified:", title, "->", result)
    time.sleep(4)
    return result

def classify_row(title):
    return get_job_category(title, client)

# Test 2 labeled rows + 2 missing rows (4 calls total, ~20 seconds)
labeled_sample = jobs[jobs["is_missing"] == False].sample(2)
missing_sample = jobs[jobs["is_missing"] == True].sample(2)
sample_jobs = pd.concat([labeled_sample, missing_sample], ignore_index=True)

sample_jobs["imputed_category"] = sample_jobs["title"].apply(classify_row)
print("\n--- Sub-Question 2: Sample Rows Classified by Gemini ---")
print(sample_jobs[["country", "title", "category_label", "imputed_category"]])

Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Classified: Capo Officina Bergamo -> other / specialized
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Classified: LGV 2 Driver -> other / specialized
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Classified: Chauffeur PL/SPL caces GRUE F/H -> other / specialized
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Gemini server busy (503), retrying in 5 seconds...
Classified: ADDETTO/A A PULIZIE PART-TIME 25 ORE SETTIMANALI -> other / specialized

--- Sub-Question 2: Sample Rows Classified by Gemini ---
          country                                             title  \
0  

In [ ]:
# Step 4: Sub-Question 2 — Check match rate on labeled rows and see hidden jobs (Weeks 4-6, 10)
def check_accuracy_and_hidden(sample_df):
    labeled = sample_df[sample_df["is_missing"] == False]
    orig_list = list(labeled["category_label"])
    pred_list = list(labeled["imputed_category"])

    matches = 0
    for i in range(len(labeled)):
        orig = str(orig_list[i]).strip().lower()
        pred = str(pred_list[i]).strip().lower()
        first_word = pred.split(" ")[0]
        if first_word in orig:
            matches += 1

    if len(labeled) > 0:
        pct = round((matches / len(labeled)) * 100, 1)
        print(f"Match rate on labeled rows: {matches}/{len(labeled)} ({pct}%)")

    missing_only = sample_df[sample_df["is_missing"] == True]
    print("\n--- Jobs Uncovered Inside Missing Sample Rows ---")
    print(missing_only["imputed_category"].value_counts())

check_accuracy_and_hidden(sample_jobs)

Match rate on labeled rows: 0/2 (0.0%)

--- Jobs Uncovered Inside Missing Sample Rows ---
imputed_category
other / specialized    2
Name: count, dtype: int64


In [ ]:
# Step 5: Outside Tool #2 — Currency Web API, USD salaries, Before vs After, and Summary (Weeks 10-12)
import requests

def fetch_eur_rates():
    url = "https://api.frankfurter.app/latest"
    params = {"from": "EUR", "to": "USD,GBP,CHF"}
    resp = requests.get(url, params=params)
    print("Currency API status:", resp.status_code)
    if resp.status_code == 200:
        return resp.json()["rates"]
    return {"USD": 1.08, "GBP": 0.85, "CHF": 0.94}

def convert_salaries_to_usd(jobs_df, analyzer, rates):
    eur_to_usd = rates.get("USD", 1.08)
    eur_to_gbp = rates.get("GBP", 0.85)
    eur_to_chf = rates.get("CHF", 0.94)
    means = jobs_df.groupby("country")["salary_mid"].mean()
    euro_countries = ["Germany", "France", "Spain", "Italy", "Netherlands"]

    print("\n--- Sub-Question 3: Mean Salary in USD ---")
    for country in analyzer.countries:
        if country in means:
            local_avg = means[country]
            if country == "USA":
                usd_avg = local_avg
            elif country == "United Kingdom":
                usd_avg = (local_avg / eur_to_gbp) * eur_to_usd
            elif country == "Switzerland":
                usd_avg = (local_avg / eur_to_chf) * eur_to_usd
            elif country in euro_countries:
                usd_avg = local_avg * eur_to_usd
            else:
                usd_avg = local_avg
            print(f"{country}: ${round(usd_avg)} USD")

def build_final_categories(sample_df):
    final_list = []
    missing_flags = list(sample_df["is_missing"])
    old_labels = list(sample_df["category_label"])
    new_labels = list(sample_df["imputed_category"])

    for i in range(len(sample_df)):
        if missing_flags[i] == True:
            final_list.append(new_labels[i])
        else:
            final_list.append(str(old_labels[i]).strip().lower())

    sample_df["final_category"] = final_list
    print("\n--- Before Imputation ---")
    print(sample_df["category_label"].value_counts())
    print("\n--- After Imputation ---")
    print(sample_df["final_category"].value_counts())
    return sample_df

rates = fetch_eur_rates()
print("Live EUR rates:", rates)
convert_salaries_to_usd(jobs, analyzer, rates)
sample_jobs = build_final_categories(sample_jobs)

# Week 12 "\n".join() summary for missing German jobs
de_jobs = jobs[jobs["country"] == "Germany"]
de_missing = de_jobs[de_jobs["is_missing"] == True]
if len(de_missing) > 0:
    time.sleep(4)
    titles_text = "\n".join(de_missing.head(10)["title"])
    summary = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="In 3 bullets, what jobs are these and what should a job board do?\n" + titles_text,
    )
    print("\n--- Gemini Business Summary (Germany) ---")
    print(summary.text)

sample_jobs.to_csv("classified_jobs.csv", index=False)
print("\nSaved classified_jobs.csv")

Currency API status: 200
Live EUR rates: {'CHF': 0.9243, 'GBP': 0.8329, 'USD': 1.0968}

--- Sub-Question 3: Mean Salary in USD ---
Germany: $25352 USD
Switzerland: $37342 USD
France: $135153 USD
Spain: $7699 USD
Netherlands: $37310 USD
Italy: $20878 USD
United Kingdom: $51890 USD
USA: $74164 USD

--- Before Imputation ---
category_label
Ingegneria                    1
Logistics & Warehouse Jobs    1
Unknown                       1
Altro                         1
Name: count, dtype: int64

--- After Imputation ---
final_category
other / specialized           2
ingegneria                    1
logistics & warehouse jobs    1
Name: count, dtype: int64


In [11]:
# Step 6: Final Business Recommendation (Weeks 3-6)
def make_recommendation(analyzer):
    priority = []
    top_country = ""
    top_pct = -1.0

    for country in analyzer.countries:
        pct = analyzer.missing_rate(country)
        if pct > top_pct:
            top_pct = pct
            top_country = country
        if pct >= 15.0:
            priority.append(f"{country} ({pct}%)")

    print("=== FINAL RECOMMENDATION ===")
    print(f"1. Biggest blind spot: {top_country} ({top_pct}% missing)")
    print("2. Priority markets (15%+ missing):", ", ".join(priority))
    print("3. Action: Auto-tag incoming European titles with Gemini before running search ads and recruiter packages.")

make_recommendation(analyzer)

=== FINAL RECOMMENDATION ===
1. Biggest blind spot: Germany (86.8% missing)
2. Priority markets (15%+ missing): Germany (86.8%), Switzerland (71.0%), France (65.0%), Spain (64.6%), Netherlands (56.2%), Italy (31.8%)
3. Action: Auto-tag incoming European titles with Gemini before running search ads and recruiter packages.
